In [1]:
with open("wikitext_10M.txt", "r", encoding="utf-8") as f:
    faqs = f.read()

print("Characters:", len(faqs))
print(faqs[:500])

FileNotFoundError: [Errno 2] No such file or directory: 'wikitext_10M.txt'

In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.preprocessing.text import Tokenizer

I0000 00:00:1781288762.646642    3958 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781288765.248050    3958 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
tokenizer = Tokenizer()

In [ ]:
tokenizer = Tokenizer(
    num_words=5000,
    oov_token="<UNK>"
)

In [ ]:
tokenizer.fit_on_texts([faqs])

In [ ]:
input_sequences = []

MAX_LEN = 21

for sentence in faqs.split('\n'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokenized_sentence)):
        seq = tokenized_sentence[max(0, i + 1 - MAX_LEN):i + 1]
        input_sequences.append(seq)

In [ ]:
MAX_LEN = max([len(x) for x in input_sequences])

In [ ]:
print(MAX_LEN)

21


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = MAX_LEN, padding='pre')

In [ ]:
X = padded_input_sequences[:,:-1]

In [ ]:
y = padded_input_sequences[:,-1]

In [ ]:
input_sequences.append(tokenized_sentence[:i+1])

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary:", vocab_size)
print("X shape:", X.shape)
print("y shape:", y.shape)

Vocabulary: 151475
X shape: (8577915, 20)
y shape: (8577915,)


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

  

checkpoint = ModelCheckpoint(
    "best_next_word.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
) 

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)   

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input

vocab_size = len(tokenizer.word_index) + 1
sequence_length = X.shape[1]

model = Sequential([
    Input(shape=(sequence_length,)),
    Embedding(input_dim=1224, output_dim=64),
    BatchNormalization(),
    LSTM(64),
    Dense(228, activation='softmax')
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 64)         │        78,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 228)            │        14,820 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,180 (492.89 KB)

 Trainable params: 126,180 (492.89 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X, y,
    epochs=5,
    batch_size=512,
    validation_split=0.2,
    callbacks = 
    [early_stop,
    reduce_lr,
    checkpoint]
)

Epoch 1/15
13399/13403 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0000e+00 - loss: nan

In [ ]:
import numpy as np
import time
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

text = "what is the fee"

for i in range(10):
    token_text = tokenizer.texts_to_sequences([text])[0]
    padded_token_text = pad_sequences([token_text], maxlen=X.shape[1], padding='pre')

    pred = model.predict(padded_token_text, verbose=0)
    pos = np.argmax(pred, axis=1)[0]

    for word, index in tokenizer.word_index.items():
        if index == pos:
            text += " " + word
            print(text)
            time.sleep(1)
            break